# Practical 02: Data Hygiene: Missing Value Imputation, Deduplication & Renaming

**UNIT 1: Introduction to Big Data and Large Dataset Handling**

**Problem Statement:** Write a Python program to perform data hygiene on a dataset with missing values and duplicate rows. Identify missing entries using isna().sum(), demonstrate both dropping rows (dropna()) and imputing missing numerical values with fillna(mean/median), remove duplicate records using drop_duplicates(), and rename columns using rename().

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None, "display.width", 200)

rs = np.random.default_rng(11)
n = 200
data = pd.DataFrame({
    "Order ID": np.arange(1001, 1001 + n),
    "Customer Name": [f"Cust-{i:03d}" for i in rs.integers(1, 80, n)],
    "City": rs.choice(["Kochi", "Chennai", "Mumbai", "Bengaluru", "Hyderabad", "Pune"], n),
    "Product Category": rs.choice(["Electronics", "Furniture", "Groceries", "Sports", "Books"], n),
    "Quantity": rs.integers(1, 8, n).astype(float),
    "Unit Price": np.round(rs.uniform(200, 6000, n), 2),
    "Discount %": rs.choice([0, 10, 15, 20, 30], n).astype(float),
    "Order Date": pd.date_range("2025-06-01", periods=n, freq="D").strftime("%Y-%m-%d"),
})
for col, k in [("Quantity", 14), ("Unit Price", 11), ("Discount %", 10), ("City", 9)]:
    data.loc[rs.choice(n, k, replace=False), col] = np.nan
data = pd.concat([data, data.sample(15, random_state=6)])
data = data.sample(frac=1, random_state=6).reset_index(drop=True)

print("Original shape:", data.shape)
print(data.head())

print(data.isna().sum())

df_nona = data.dropna()
print(f"dropna(): {data.shape[0]} -> {df_nona.shape[0]} rows ({data.shape[0] - df_nona.shape[0]} rows lost)")

df_filled = data.copy()
df_filled["Quantity"] = df_filled["Quantity"].fillna(df_filled["Quantity"].median())
df_filled["Unit Price"] = df_filled["Unit Price"].fillna(df_filled["Unit Price"].mean())
df_filled["Discount %"] = df_filled["Discount %"].fillna(df_filled["Discount %"].mean())
df_filled["City"] = df_filled["City"].fillna(df_filled["City"].mode()[0])

print("Missing values after imputation:")
print(df_filled.isna().sum())

print("Duplicate rows found:", df_filled.duplicated().sum())
df_final = df_filled.drop_duplicates().reset_index(drop=True)
print("Shape after drop_duplicates():", df_final.shape)

df_final = df_final.rename(columns={
    "Order ID": "OrderID",
    "Customer Name": "CustomerName",
    "City": "City",
    "Product Category": "ProductCategory",
    "Quantity": "Quantity",
    "Unit Price": "UnitPrice",
    "Discount %": "DiscountPct",
    "Order Date": "OrderDate",
})
print(df_final.columns.tolist())
print(df_final.head())

Original shape: (215, 8)

,Order ID,Customer Name,City,Product Category,Quantity,Unit Price,Discount %,Order Date
0,1084,Cust-016,Mumbai,Furniture,2.0,3535.03,20.0,2025-08-23
1,1007,Cust-057,Hyderabad,Electronics,1.0,4312.02,30.0,2025-06-07
2,1108,Cust-003,Chennai,Groceries,4.0,4761.66,30.0,2025-09-16
3,1195,Cust-045,Mumbai,Books,7.0,1978.53,10.0,2025-12-12
4,1171,Cust-062,Kochi,Electronics,6.0,NaN,10.0,2025-11-18


Order ID             0
Customer Name        0
City                 9
Product Category     0
Quantity            15
Unit Price          12
Discount %          11
Order Date           0
dtype: int64
dropna(): 215 -> 174 rows (41 rows lost)
Missing values after imputation:
Order ID            0
Customer Name       0
City                0
Product Category    0
Quantity            0
Unit Price          0
Discount %          0
Order Date          0
dtype: int64
Duplicate rows found: 15
Shape after drop_duplicates(): (200, 8)
['OrderID', 'CustomerName', 'City', 'ProductCategory', 'Quantity', 'UnitPrice', 'DiscountPct', 'OrderDate']

,OrderID,CustomerName,City,ProductCategory,Quantity,UnitPrice,DiscountPct,OrderDate
0,1084,Cust-016,Mumbai,Furniture,2.0,3535.030000,20.0,2025-08-23
1,1007,Cust-057,Hyderabad,Electronics,1.0,4312.020000,30.0,2025-06-07
2,1108,Cust-003,Chennai,Groceries,4.0,4761.660000,30.0,2025-09-16
3,1195,Cust-045,Mumbai,Books,7.0,1978.530000,10.0,2025-12-12
4,1171,Cust-062,Kochi,Electronics,6.0,3226.085123,10.0,2025-11-18
